# Phase 5: Dimensionality Reduction

This notebook applies PCA and UMAP to visualize the hidden state trajectories of the Transformer representations. We generate publication-quality static figures using Matplotlib and interactive plots using Plotly, saving output metadata and coordinates for Phase 6 metrics.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px

# Add parent directory to path to allow src imports
sys.path.append(os.path.dirname(os.getcwd()))
from src.dimensionality_reduction import fit_global_models, project_and_save

## 1. Fit Global Dimensionality Reduction Models

In [ ]:
MODEL_NAME = "gpt2"
# Set base dir appropriately given we run in notebooks/ or root
base_dir = "../" if os.path.exists("../data") else "./"
# Fit global PCA(50) and UMAP(3) across all prompts for the model, saving to results/
pca_model, umap_model = fit_global_models(MODEL_NAME, trajectories_dir=base_dir+"data/trajectories", models_dir=base_dir+"results/models", diagnostics_dir=base_dir+"results/diagnostics")
# Project and save parquet coordinates with metadata
df_pca, df_umap = project_and_save(MODEL_NAME, pca_model, umap_model, prompts_file=base_dir+"data/prompts/prompts.jsonl", trajectories_dir=base_dir+"data/trajectories", projected_dir=base_dir+"results/projected")

## 2. Load Diagnostics (Variance and Trustworthiness)

In [ ]:
# Plot PCA Explained Variance
with open(f"{base_dir}results/diagnostics/{MODEL_NAME}_pca_variance.json", "r") as f:
    pca_diag = json.load(f)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pca_diag['cumulative_variance']) + 1), pca_diag['cumulative_variance'], marker='o')
plt.title(f"{MODEL_NAME} PCA Cumulative Explained Variance")
plt.xlabel("Principal Component")
plt.ylabel("Cumulative Variance Ratio")
plt.grid(True)
plt.savefig(f"{base_dir}figures/paper/{MODEL_NAME}_pca_variance.png", dpi=300)
plt.show()

# Show UMAP Trustworthiness
with open(f"{base_dir}results/diagnostics/{MODEL_NAME}_umap_trustworthiness.json", "r") as f:
    umap_diag = json.load(f)
print(f"UMAP Trustworthiness: {umap_diag['trustworthiness']:.4f}")

## 3. Visualization Helpers

In [ ]:
def plot_matplotlib_3d(df, title, filename=None, group_by_prompt=True):
    """
    Static publication-quality 3D plot.
    Uses a colormap to represent depth (temporal progression across layers).
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    if group_by_prompt:
        prompts = df['prompt_id'].unique()
        for p_id in prompts:
            p_df = df[df['prompt_id'] == p_id].sort_values('layer')
            sc = ax.scatter(p_df['x'], p_df['y'], p_df['z'], 
                            c=p_df['layer'], cmap='viridis', s=20, alpha=0.8, edgecolor='k')
            ax.plot(p_df['x'], p_df['y'], p_df['z'], color='gray', alpha=0.5, linewidth=1)
    else:
        # For centroid plotting, there is only one trajectory
        df = df.sort_values('layer')
        sc = ax.scatter(df['x'], df['y'], df['z'], 
                        c=df['layer'], cmap='plasma', s=50, alpha=1.0, edgecolor='k')
        ax.plot(df['x'], df['y'], df['z'], color='black', alpha=0.8, linewidth=2)

    ax.set_title(title)
    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')
    ax.set_zlabel('Dim 3')
    plt.colorbar(sc, label='Layer (Depth)')
    
    if filename:
        plt.savefig(filename, dpi=300)
    plt.show()

def plot_plotly_3d(df, title, group_by_prompt=True):
    """
    Interactive 3D Plotly plot.
    """
    if group_by_prompt:
        fig = px.line_3d(df, x='x', y='y', z='z', color='prompt_id', 
                         markers=True, title=title, 
                         hover_data=['group', 'subcategory', 'layer'])
    else:
        fig = px.line_3d(df, x='x', y='y', z='z', color='group',
                         markers=True, title=title, 
                         hover_data=['layer'])
        
    # Enhance markers with temporal colors
    fig.update_traces(marker=dict(size=4, 
                                  color=df['layer'] if not group_by_prompt else None, 
                                  colorscale='Viridis', showscale=False))
    fig.show()

## 4. Figures: Trajectories

In [ ]:
# Load data
df_pca = pd.read_parquet(f"{base_dir}results/projected/{MODEL_NAME}_pca.parquet")
df_umap = pd.read_parquet(f"{base_dir}results/projected/{MODEL_NAME}_umap.parquet")

for method, df in [('PCA', df_pca), ('UMAP', df_umap)]:
    # 1. Figure 2A: Single Prompt Trajectory
    single_prompt_id = df['prompt_id'].iloc[0]
    single_df = df[df['prompt_id'] == single_prompt_id]
    plot_matplotlib_3d(single_df, f"{method}: Single Trajectory ({single_prompt_id})", 
                       f"{base_dir}figures/paper/{MODEL_NAME}_{method.lower()}_single_traj.png")
    
    # 2. Figure 2B: Family Overlay (e.g., animals)
    animals_df = df[df['group'] == 'animals']
    plot_matplotlib_3d(animals_df, f"{method}: Animals Family Overlay", 
                       f"{base_dir}figures/{method.lower()}/animals_overlay.png")
    
    # Plotly Exploration
    plot_plotly_3d(animals_df, f"{method}: Animals Family Overlay (Interactive)")
    
    # 3. Figure 2C: Family Centroid Trajectories
    # Compute centroid by averaging (x,y,z) for each layer across all prompts in the group
    centroid_df = animals_df.groupby('layer')[['x', 'y', 'z']].mean().reset_index()
    centroid_df['group'] = 'animals_centroid'
    
    plot_matplotlib_3d(centroid_df, f"{method}: Animals Family Centroid", 
                       f"{base_dir}figures/paper/{MODEL_NAME}_{method.lower()}_animals_centroid.png", group_by_prompt=False)
